In [1]:
# Tensorflow has lots of warnings that I don't like. This part forces the code ignore them.
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # 0=all, 1=filter INFO, 2=filter WARNING, 3=filter ERROR too

# Use the second GPU.
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

import tensorflow as tf
from resnet_backend import *
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.layers import Dense, Add, Input, Activation, Lambda
from sklearn.metrics import confusion_matrix,accuracy_score

In [2]:

def lr_schedule(epoch):
    """Learning Rate Schedule

    Learning rate is scheduled to be reduced after 80, 120, 160, 180 epochs.
    Called automatically every epoch as part of callbacks during training.

    # Arguments
        epoch (int): The number of epochs

    # Returns
        lr (float32): learning rate
    """
    lr = 1e-3
    if epoch > 180:
        lr *= 0.5e-3
    elif epoch > 160:
        lr *= 1e-3
    elif epoch > 120:
        lr *= 1e-2
    elif epoch > 80:
        lr *= 1e-1
    print('Learning rate: ', lr)
    return lr


def get_callbacks(model_type):
    save_dir = os.path.join(os.getcwd(), 'Saved_Models/')
    model_name = "%s_model.{epoch:03d}.keras" % model_type
    if not os.path.isdir(save_dir):
        os.makedirs(save_dir)
    filepath = os.path.join(save_dir, model_name)

    # Prepare callbacks for model saving and for learning rate adjustment.
    checkpoint = ModelCheckpoint(
        filepath=filepath,
        monitor='val_accuracy',
        verbose=1,
        save_best_only=True,mode='max')

    lr_scheduler = LearningRateScheduler(lr_schedule)

    lr_reducer = ReduceLROnPlateau(
        factor=np.sqrt(0.1),
        cooldown=0,
        patience=5,
        min_lr=0.5e-6)

    return [checkpoint, lr_reducer, lr_scheduler]    

In [3]:

def create_model_resnet(input_shape):
    n = 3
    depth = n * 9 + 2
    model_type = 'Fuzzy-ResNet%dv%d-trial-1' % (depth, 2)
    inputs, features = resnet_backend_v2(
        input_shape=input_shape,
        depth=depth)
    memberships = LogGaussMF(10)(features)
    rules = Lambda(lambda x: tf.keras.ops.sum(x, axis=-1),output_shape=(10,))(memberships)
    linear = Dense(10)(features)
    logits = Add()([rules, linear])
    outputs = Activation("softmax")(logits)
    model = Model(inputs=inputs, outputs=outputs)

    return model

In [4]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()


x_train=np.resize(x_train,(60000,28,28,1))
x_test=np.resize(x_test,(10000,28,28,1))

# Input image dimensions.
input_shape = x_train.shape[1:]

# Normalize data.
x_train = x_train.astype('float32') / 255
x_test = x_test.astype('float32') / 255



# If subtract pixel mean is enabled
x_train_mean = np.mean(x_train, axis=0)
x_train -= x_train_mean
x_test -= x_train_mean


print('x_train shape:', x_train.shape)
print(x_train.shape[0], 'train samples')
print(x_test.shape[0], 'test samples')
print('y_train shape:', y_train.shape)

x_train shape: (60000, 28, 28, 1)
60000 train samples
10000 test samples
y_train shape: (60000,)


In [5]:
model=create_model_resnet(input_shape)

model.compile(
    loss='categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(),
    metrics=['accuracy'])

In [6]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 28, 28, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 28, 28,    │        160 │ input_layer[0][0] │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 28, 28,    │         64 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 28, 28,    │          0 │ batch_normalizat… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 28, 28,    │        272 │ activation[0][0]  │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │         64 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 28, 28,    │          0 │ batch_normalizat… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 28, 28,    │      2,320 │ activation_1[0][… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │         64 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 28, 28,    │          0 │ batch_normalizat… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 28, 28,    │      1,088 │ activation[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 28, 28,    │      1,088 │ activation_2[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 28, 28,    │          0 │ conv2d_4[0][0],   │
│                     │ 64)               │            │ conv2d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │        256 │ add[0][0]         │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 28, 28,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 28, 28,    │      1,040 │ activation_3[0][… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │         64 │ conv2d_5[0][0]  

 Total params: 853,834 (3.26 MB)

 Trainable params: 848,618 (3.24 MB)

 Non-trainable params: 5,216 (20.38 KB)

In [7]:
model_type = "DCNFIS_ResNetV2_n3"
callbacks = get_callbacks(model_type)

In [8]:
# (x_train, y_train), (x_test, y_test)
model.fit(x_train, y_train,
          batch_size=128,
          epochs=20,
          validation_data=(x_test, y_test),
          shuffle=True,
          callbacks=callbacks)

Learning rate:  0.001
Epoch 1/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.9345 - loss: 0.7262
Epoch 1: val_accuracy improved from None to 0.83540, saving model to /home/mojtaba/Desktop/TF_Works/Thesis_Rep/Saved_Models/DCNFIS_ResNetV2_n3_model.001.keras

Epoch 1: finished saving model to /home/mojtaba/Desktop/TF_Works/Thesis_Rep/Saved_Models/DCNFIS_ResNetV2_n3_model.001.keras
469/469 ━━━━━━━━━━━━━━━━━━━━ 42s 49ms/step - accuracy: 0.9344 - loss: 0.7262 - val_accuracy: 0.8354 - val_loss: 0.8841 - learning_rate: 0.0010
Learning rate:  0.001
Epoch 2/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9841 - loss: 0.3930
Epoch 2: val_accuracy improved from 0.83540 to 0.94180, saving model to /home/mojtaba/Desktop/TF_Works/Thesis_Rep/Saved_Models/DCNFIS_ResNetV2_n3_model.002.keras

Epoch 2: finished saving model to /home/mojtaba/Desktop/TF_Works/Thesis_Rep/Saved_Models/DCNFIS_ResNetV2_n3_model.002.keras
469/469 ━━━━━━━━━━━━━━━━━━━━ 9s 20ms/step - accuracy: 0.9841 - loss:

In [10]:
model.load_weights('./Saved_Models/DCNFIS_ResNetV2_n3_model.018.keras')
score = model.evaluate(x_test, y_test, verbose=0)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

Test loss: 0.10089807957410812
Test accuracy: 0.9868999719619751
